# 작업 노트북: 학습 → 제출 zip → 리허설 → 제출

이 노트북은 **내 PC에서만** 실행합니다. 채점 서버로 올라가는 것은 `dist/<이름>/predict.ipynb` 입니다.

`aifactory submit` 은 **셀을 실행한 노트북 자체**를 predict.ipynb 로 올립니다. 그래서 이 노트북에서 제출할 때는
반드시 `--notebook predict.ipynb` 로 제출용 노트북을 지정하고, 작업 폴더를 `dist/<이름>/` 으로 옮겨서 실행합니다
(그 폴더의 `requirements.txt` 와 `assets/` 가 함께 올라갑니다).

참여키는 노트북에 적지 않습니다. 환경변수 `AIF_API_KEY` 를 설정하거나, 제출 셀 실행 때 뜨는 입력창에 넣으십시오.

In [ ]:
# [설정] 필요한 것만 고치십시오
import os, sys, subprocess, json
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)                                      # change_detection/ 에서 모든 명령 실행
BASE = ROOT / "submission/assets/model/unet_r18_cd.pt"   # 주최 측 베이스라인 가중치 (git 제외)

NAME = "baseline-v1"        # 제출 모델명 = dist/<NAME>/
MODELS = [BASE]             # 앙상블할 가중치 목록. 학습 후에는 [ROOT/"runs/r18_cls/best.pt"] 등
RULES = None                # tune_rules.py 결과 json 경로. None 이면 베이스라인과 같은 기본 규칙
TTA = 1                     # 1 = 끔 (베이스라인과 같음)

RUN_TRAIN = False           # True 로 바꾸면 아래 학습 셀이 실제로 돕니다
TRAIN_SOURCES = ["data/aihub_cd:2", "data/synth_cd:1"]
VAL_DIR = "data/val"
REHEARSAL_INPUT = None      # 리허설 입력 폴더 (pairs.csv + images/<id>/pre.png, post.png). None 이면 건너뜀

def run(*args):
    print("$", " ".join(map(str, args)))
    subprocess.run([sys.executable, *map(str, args)], check=True)

import torch
print("python", sys.version.split()[0], "| torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| gpu", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
print("베이스라인 가중치:", "있음" if BASE.exists() else "없음 -> submission/assets/model/ 에 넣으십시오")

In [ ]:
# [학습] (선택) 베이스라인에서 출발하는 파인튜닝. 매 epoch 검증 점수가 출력됩니다
if RUN_TRAIN:
    run("scripts/train.py", "--train", *TRAIN_SOURCES, "--val", VAL_DIR, "--baseline", BASE,
        "--cls-head", "--epochs", 30, "--out", "runs/r18_cls")
    run("scripts/predict_folder.py", "--ckpt", "runs/r18_cls/best.pt", "--pairs", VAL_DIR, "--out", "runs/val.npz")
    run("scripts/tune_rules.py", "--probs", "runs/val.npz", "--val", VAL_DIR, "--out", "runs/rules.json")
else:
    print("학습 건너뜀 (RUN_TRAIN=False)")

In [ ]:
# [제출 zip 만들기] dist/<NAME>/ 폴더와 dist/<NAME>.zip
args = ["scripts/build_submission.py", "--name", NAME, "--tta", TTA, "--models", *MODELS]
if RULES:
    args += ["--rules", RULES]
run(*args)
SUB = ROOT / "dist" / NAME
print(sorted(str(p.relative_to(SUB)) for p in SUB.rglob("*") if p.is_file() and "cdkit" not in p.parts))

In [ ]:
# [리허설] 서버처럼 제출 폴더 안에서 predict.ipynb 를 실행하고 결과 파일 규칙을 검사
if REHEARSAL_INPUT:
    inp = Path(REHEARSAL_INPUT).resolve()
    out = ROOT / "runs" / f"rehearsal_{NAME}.csv"
    env = {**os.environ, "AIF_INPUT_DIR": str(inp), "AIF_PREDICTION_PATH": str(out)}
    subprocess.run([sys.executable, "-m", "jupyter", "nbconvert", "--to", "notebook", "--execute",
                    "predict.ipynb", "--output", str(ROOT / "runs" / f"rehearsal_{NAME}.ipynb")],
                   cwd=SUB, env=env, check=True)
    run("scripts/validate_submission.py", out, "--pairs", inp / "pairs.csv")
else:
    print("리허설 건너뜀 (REHEARSAL_INPUT=None)")

## 제출

- `DEBUG = True`: 주최 측 공개 디버그 데이터로 실행되고 **실행 로그를 제출 이력에서 볼 수 있습니다.** 리더보드에 반영되지 않고 하루 횟수 제한이 있습니다. 새 코드는 먼저 디버그로 확인하십시오.
- `DEBUG = False`: 실제 제출 (리더보드 반영).
- 참여키: 환경변수 `AIF_API_KEY` 가 없으면 입력창이 뜹니다. **노트북에 적지 마십시오.**

In [ ]:
# [제출] DO_SUBMIT 을 True 로 바꿔야 실제로 올라갑니다
DO_SUBMIT = False
DEBUG = True

if DO_SUBMIT:
    %load_ext aifactory
    os.chdir(SUB)                       # 이 폴더의 requirements.txt 와 assets/ 가 함께 올라갑니다
    try:
        flag = "--debug" if DEBUG else ""
        %aifactory submit --notebook predict.ipynb --model-name {NAME} {flag}
    finally:
        os.chdir(ROOT)
else:
    print(f"제출하지 않음. 올라갈 폴더: {SUB}")